# Getting Started with Local AI

Let's get your computer set up to test out product ideas without needing any API keys. :) There are plenty of freely available local Ai models that can help you develop minimal MVPs without paying for a single token.

In this notebook, we will:

1. Get [ollama](https://ollama.com/) set up and running. You might also want to [download the desktop-based software](https://ollama.com/download) in case you want to test out prompts outside of these notebooks.

2. Get at least one [llamafile](https://github.com/Mozilla-Ocho/llamafile) running. The [QuickStart](https://docs.mozilla.ai/llamafile/getting-started/quickstart) can guide you step by step.

3. Set up a workflow for loading and saving prompts and responses.

In [ ]:
import ollama

## Setting up an ollama client

If you are running in a devcontainer, ollama is already running in another docker. That means you can use the next cell to set up a client. 

In [ ]:
# If using a devcontainer, use this to set up your client.
client = ollama.Client('http://ollama:11434')

**Do not** run the following cell if you are using a devcontainer.

The next cell is for people running the Jupyter notebook natively and who have installed the `ollama` Python library on their system.

In [ ]:
# Run this code to set up the client when running Jupyter natively
# Do not run if you are using a devcontainer :) 
client = ollama

## List models available to ollama

In [ ]:
for model in client.list().models:
    print(model.model)

## Explore and Download a Model

Check out and download at least one model from [the list of available models](https://ollama.com/library?sort=popular). Note: if you do not have a GPU or M1+ chip, you probably should start with the smaller models (i.e. 4b or less).

In [ ]:
model_name = 'gemma3:4b'

In [ ]:
client.pull(model_name)

## Prompt the model

Try it out!

This sets up a `test_prompt` and forms `messages` for the client to send to the model for a response.

In [ ]:
# Feel free to experiment and change the test_prompt
test_prompt = "ummmm... anyone there?"

In [ ]:
messages = [{
    'role': 'user',
    'content': test_prompt
}]

In [ ]:
response = client.chat(
    model=model_name,
    messages=messages
)

### Review the model's response

In [ ]:
# The raw response
response

In [ ]:
# Display a dictionary of response information and message
response.__dict__

In [ ]:
# Display the response's message
print(response.message.content)

### Clean up and delete the ollama client

In [ ]:
del client

## Let's get llamafile going

1. Follow the [llamafile quickstart](https://mozilla-ai.github.io/llamafile/quickstart/) using a new terminal window.
2. Complete steps 1-4 in the llamafile quickstart.
   You will see the llamafile chat in the terminal.
3. Navigate to http://localhost:8080/ to open llama.cpp in your browser.
   It will have lots of options to enter a prompt.
   You can test it in the browser at the bottom of the opened page.

🛑 **Do the steps above before continuing.**

In [ ]:
from openai import OpenAI

In [ ]:
# If using a devcontainer, run this cell and skip the following cell.
base_url="http://llamafile:8080/v1"

In [ ]:
# If running locally, run this cell.
base_url="http://localhost:8080/v1"

In [ ]:
client = OpenAI(
    base_url=base_url, # Sets "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)

In [ ]:
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=messages
)

### Review the response from a different model

This time we sent the messages to a different model than we used in the above section.

In [ ]:
completion

In [ ]:
completion.__dict__

In [ ]:
completion.choices[0].message.content

### Conclusions from the response

Well, we can see not all models are built alike ;)

## Build a prompt template, use it, and save chats

We will set up:

1. a very simple meta prompt template
2. an example prompt
3. an example conversation

The point is to give us some conventions around saving our work as we progress and building best practices to save conversations for later evaluation and review.

In [ ]:
import json

In [ ]:
meta_prompt_template = """
{role_description}

{task_description}

{rules}

{examples}

{context}
"""


In [ ]:
role = "You are a helpful assistant who makes tasty recipes."
task = """You use regular pantry items and things everyone has 
in their kitchen and help them make new, inventive recipes with those ingredients."""
rules = "You do not use fancy ingredients that people might not have access to use."
examples = "An example recipe would be a French omlette with spruced up canned beans and a side salad with a quirky vinegrette."
context = "You take inspiration from Nigella Lawson."

In [ ]:
meta_prompt_template.format(
    role_description=role,
    task_description=task,
    rules=rules,
    examples=examples,
    context=context)

In [ ]:
with open('data/templates/meta_prompt_template', 'w') as mpt:
    mpt.write(meta_prompt_template)

In [ ]:
with open('data/meta_prompts/pantry_chef_prompt', 'w') as cpt:
    cpt.write(meta_prompt_template.format(
        role_description=role,
        task_description=task,
        rules=rules,
        examples=examples,
        context=context))

In [ ]:
meta_prompt = meta_prompt_template.format(
    role_description=role,
    task_description=task,
    rules=rules,
    examples=examples,
    context=context)

In [ ]:
conversation = [
    {'role': 'system', 
     'content': meta_prompt},
    {'role': 'user',
     'content': """Help! I only have canned tomatoes, some chicken and an onion.... 
     what can I make for dinner?"""},
]

In [ ]:
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=conversation
)

In [ ]:
completion

In [ ]:
completion.choices[0].message.content

In [ ]:
conversation.append({
    'role': 'agent',
    'content': completion.choices[0].message.content
})

In [ ]:
with open('data/traces/chef_chicken_pasta.json', 'w') as chef_chat:
    json.dump(conversation, chef_chat)

In [ ]:
from pprint import pprint

In [ ]:
pprint(conversation)